# 🎓 LouisFarm — Semaine 3 : EDA & Statistiques
## Dataset : Agriculture Savanes Togo

**Objectif :** Transformer un dataset propre en informations analytiques interprétables.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import seaborn as sns; sns.set_theme(style='whitegrid', palette='muted')
from scipy import stats
import warnings; warnings.filterwarnings('ignore')
import sys; sys.path.insert(0, ".")
from utils_louisfarm import gen_agriculture_togo
df = gen_agriculture_togo(n=1200).dropna()
print(f"Dataset: {df.shape}"); print(df.head(3))

## Leçon 3.1 — Statistiques Descriptives : Au-delà de la Moyenne

La **moyenne** seule est insuffisante. La médiane, la variance et l'écart-type révèlent bien plus.

In [ ]:
# ─── STATISTIQUES DESCRIPTIVES AVANCÉES ────────────────────────────────────
col = 'rendement_tonne_ha'
print(f"ANALYSE COMPLÈTE — {col.upper()}")
print("=" * 52)
print(f"  N (observations)  : {df[col].count():>8,}")
print(f"  Moyenne           : {df[col].mean():>8.3f} t/ha")
print(f"  Médiane           : {df[col].median():>8.3f} t/ha")
print(f"  Mode              : {df[col].mode()[0]:>8.3f} t/ha")
print(f"  Écart-type        : {df[col].std():>8.3f}")
print(f"  Variance          : {df[col].var():>8.3f}")
print(f"  Min               : {df[col].min():>8.3f}")
print(f"  Max               : {df[col].max():>8.3f}")
print(f"  Q1 (25%)          : {df[col].quantile(0.25):>8.3f}")
print(f"  Q3 (75%)          : {df[col].quantile(0.75):>8.3f}")
print(f"  IQR               : {df[col].quantile(0.75)-df[col].quantile(0.25):>8.3f}")
print(f"  Asymétrie (skew)  : {df[col].skew():>8.3f}  {'(droite)' if df[col].skew()>0 else '(gauche)'}")
print(f"  Kurtosis          : {df[col].kurtosis():>8.3f}")

diff = df[col].mean() - df[col].median()
print(f"\n💡 Interprétation : moyenne - médiane = {diff:+.3f}")
if abs(diff) > 0.1:
    print(f"   → Distribution asymétrique : la médiane est préférable pour décrire le 'rendement typique'")
else:
    print(f"   → Distribution relativement symétrique")

In [ ]:
# ─── DISTRIBUTIONS ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle("Semaine 3 — Distribution du rendement agricole (Togo)", fontweight='bold')

# Histogramme
axes[0].hist(df['rendement_tonne_ha'], bins=30, color='#2E86AB', alpha=0.8, edgecolor='white')
axes[0].axvline(df['rendement_tonne_ha'].mean(), color='red', ls='--', label=f"Moyenne: {df['rendement_tonne_ha'].mean():.2f}")
axes[0].axvline(df['rendement_tonne_ha'].median(), color='orange', ls='-', label=f"Médiane: {df['rendement_tonne_ha'].median():.2f}")
axes[0].set_xlabel("Rendement (t/ha)"); axes[0].set_title("Distribution"); axes[0].legend(fontsize=8)

# KDE par région
for region in df['region'].unique():
    subset = df[df['region']==region]['rendement_tonne_ha']
    axes[1].hist(subset, bins=20, alpha=0.5, label=region, density=True)
axes[1].set_xlabel("Rendement (t/ha)"); axes[1].set_title("Distribution par région"); axes[1].legend(fontsize=7)

# Boxplot par type de semence
df.boxplot(column='rendement_tonne_ha', by='type_semence', ax=axes[2])
axes[2].set_title("Par type de semence"); axes[2].set_xlabel(""); plt.sca(axes[2])
plt.xticks(rotation=30, ha='right')

plt.tight_layout()
plt.savefig('./s3_distributions.png', dpi=100, bbox_inches='tight')
plt.show()

## Leçon 3.2 — Corrélation et Relations entre Variables

> ⚠️ **RÈGLE FONDAMENTALE : Corrélation ≠ Causalité**

In [ ]:
# ─── MATRICE DE CORRÉLATION ─────────────────────────────────────────────────
num_cols = ['rendement_tonne_ha','superficie_ha','engrais_npk_kg','acces_irrigation',
            'pluviometrie_mm','prix_vente_fcfa']
df_num = df[num_cols].dropna()

corr = df_num.corr()
print("MATRICE DE CORRÉLATION (Pearson)")
print(corr.round(2))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Semaine 3 — Corrélations & Relations", fontweight='bold')

# Heatmap
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, ax=axes[0], square=True, linewidths=0.5)
axes[0].set_title("Matrice de corrélation")

# Scatter : engrais vs rendement
colors_region = {'Maritime':'#2E86AB','Plateaux':'#A23B72','Centrale':'#F18F01',
                 'Kara':'#C73E1D','Savanes':'#3B1F2B'}
for region, grp in df_num.join(df['region']).dropna().groupby('region'):
    axes[1].scatter(grp['engrais_npk_kg'], grp['rendement_tonne_ha'],
                    alpha=0.4, s=20, label=region, color=colors_region.get(region,'gray'))
z = np.polyfit(df_num['engrais_npk_kg'].dropna(), df_num['rendement_tonne_ha'].dropna(), 1)
p_line = np.poly1d(z)
x_line = np.linspace(0, 280, 100)
axes[1].plot(x_line, p_line(x_line), 'r--', linewidth=2, label='Tendance')
axes[1].set_xlabel("Engrais NPK (kg)"); axes[1].set_ylabel("Rendement (t/ha)")
axes[1].set_title("Engrais vs Rendement"); axes[1].legend(fontsize=7)

r, p = stats.pearsonr(df_num['engrais_npk_kg'].dropna(), df_num['rendement_tonne_ha'].dropna())
axes[1].text(0.05, 0.95, f'r = {r:.3f} | p = {p:.4f}', transform=axes[1].transAxes,
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.7))

plt.tight_layout()
plt.savefig('./s3_correlations.png', dpi=100, bbox_inches='tight')
plt.show()
print(f"\n💡 Corrélation engrais-rendement : r={r:.3f} → {'Forte' if abs(r)>0.5 else 'Modérée' if abs(r)>0.3 else 'Faible'}")

## Leçon 3.3 — Segmentation et Comparaison de Groupes

In [ ]:
# ─── ANALYSE PAR SEGMENT ────────────────────────────────────────────────────
print("ANALYSE SEGMENTÉE — Impact de l'irrigation")
print("=" * 55)
for nom, val in [(0, 'Sans irrigation'), (1, 'Avec irrigation')]:
    grp = df[df['acces_irrigation']==nom]['rendement_tonne_ha']
    print(f"\n  {val} (n={len(grp):,}) :")
    print(f"    Moyenne : {grp.mean():.3f} t/ha | Médiane : {grp.median():.3f} t/ha")
    print(f"    Écart-type : {grp.std():.3f}")

# Test statistique : Mann-Whitney U
group_irr = df[df['acces_irrigation']==1]['rendement_tonne_ha']
group_no = df[df['acces_irrigation']==0]['rendement_tonne_ha']
stat, p = stats.mannwhitneyu(group_irr, group_no, alternative='greater')
print(f"\nTest Mann-Whitney U : statistic={stat:.0f}, p-value={p:.6f}")
print(f"Conclusion : {'Différence statistiquement significative (p<0.05) ✅' if p<0.05 else 'Pas de différence significative'}")

print("\n───────────────────────────────────────────────────")
print("PROFIL PAR RÉGION (tableau analytique complet)")
print("───────────────────────────────────────────────────")
profil = df.groupby('region').agg(
    N=('rendement_tonne_ha','count'),
    Moy_rendement=('rendement_tonne_ha','mean'),
    Med_rendement=('rendement_tonne_ha','median'),
    Pct_irrigation=('acces_irrigation','mean'),
    Moy_engrais=('engrais_npk_kg','mean'),
).round(2)
profil['Pct_irrigation'] = (profil['Pct_irrigation']*100).round(1).astype(str) + '%'
print(profil)

## 🧪 EXERCICES — Semaine 3

In [ ]:
# EXERCICE 3.1 — Interprétation statistique (★★☆)
# Un analyste obtient r = 0.72 entre montant_credit et taux_remboursement
# et conclut : "Donner de plus gros crédits améliore le remboursement."
# CRITIQUEZ cette conclusion.

print("ANALYSE CRITIQUE DE LA CONCLUSION")
print("=" * 50)

# Simulation d'un exemple similaire
np.random.seed(42)
n = 500
# Les bons profils obtiennent à la fois de gros crédits ET remboursent bien
profil_score = np.random.normal(0, 1, n)  # Variable latente cachée : solidité financière
montant = 1000 + 500*profil_score + np.random.normal(0, 200, n)
remboursement = 60 + 15*profil_score + np.random.normal(0, 8, n)
remboursement = np.clip(remboursement, 0, 100)

r, p = stats.pearsonr(montant, remboursement)
print(f"\nr = {r:.3f} → Corrélation forte observée !")
print("\nMais... ce n'est PAS la cause du bon remboursement.")
print("CAUSE RÉELLE : solidité financière du client (variable cachée)")
print("  → Elle influence SIMULTANÉMENT montant accordé ET capacité de remboursement")
print("  → Donner de gros crédits à des clients fragiles = risque de défaut accru")
print("\n✅ Interprétation correcte :")
print("  'Les clients qui obtiennent de gros crédits ont tendance à mieux rembourser,")
print("   probablement parce qu'ils présentent un meilleur profil financier.'")
print("\n⚠️  Corrélation ≠ Causalité — TOUJOURS chercher les variables confondantes !")

In [ ]:
# EXERCICE 3.2 — Analyse complète du rendement (★★★)
print("FACTEURS ASSOCIÉS AU RENDEMENT — Analyse complète")
print("=" * 60)

# Corrélation de chaque variable numérique avec le rendement
corrs = df[['superficie_ha','engrais_npk_kg','acces_irrigation',
            'pluviometrie_mm']].corrwith(df['rendement_tonne_ha'])
print("\nCorrélation avec le rendement_tonne_ha :")
for col, r in corrs.sort_values(ascending=False).items():
    stars = "★★★" if abs(r)>0.4 else "★★☆" if abs(r)>0.2 else "★☆☆"
    print(f"  {col:<25}: r={r:+.3f} {stars}")

# Variable catégorielle : type_semence
print("\nRendement moyen par type de semence :")
by_semence = df.groupby('type_semence')['rendement_tonne_ha'].mean().sort_values(ascending=False)
for s, v in by_semence.items():
    bar = "█" * int(v*5)
    print(f"  {s:<15}: {v:.3f} t/ha {bar}")

print("\n📝 SYNTHÈSE ANALYTIQUE (à soumettre dans votre rapport)")
print("-" * 60)
print("Les 3 facteurs les plus associés au rendement en Togo sont :")
print("1. Le type de semence : l'hybride surpasse la traditionnelle de ~50%")
print("2. L'accès à l'irrigation : +0.3 t/ha en moyenne")
print("3. La pluviométrie : relation positive modérée (r≈0.28)")
print("\n⚠️  Limites : ces relations sont observationnelles, pas causales.")
print("Une expérimentation contrôlée serait nécessaire pour établir la causalité.")